
# LSTM - AESO Price Spike Forecasting

**Target**: `spike_lead_2` - 1 if pool price at `t+2` exceeds `$200/MWh`, else 0.
We predict it using only information available at time `t`.

**Why LSTM?**
LSTMs are recurrent, so the sequence window itself carries the recent temporal history.
That means we do not need manual lag features here.

This notebook follows the structure of the reference notebook you shared, but it is adapted for this local project:
- no `google.colab`
- local repo paths
- Windows-safe dataloaders (`num_workers=0`)
- random search instead of Optuna
- F2 used directly for tuning and evaluation


In [1]:
# 1. Imports and config
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import fbeta_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

# These paths are relative to this notebook's folder: JorgeFolder/models/lstm/
SOURCE_DATA = Path("../../../Data/CSVs/aeso_merged_2020_2025.csv")
OUTPUT_DIR = Path("random_search_run")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

if not SOURCE_DATA.exists():
    raise FileNotFoundError("Could not resolve ../../../Data/CSVs/aeso_merged_2020_2025.csv from the notebook folder.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "spike_lead_2"
TRAIN_END_EXCLUSIVE = pd.Timestamp("2023-11-06 00:00:00")
VAL_END_EXCLUSIVE = pd.Timestamp("2024-12-12 00:00:00")
TEST_START = VAL_END_EXCLUSIVE
N_CV_SPLITS = 5
MAX_EPOCHS = 10
PATIENCE = 3

# We keep 0.50 only as a quick reference point during training.
# For model selection and final evaluation, thresholds are tuned on validation.
THRESHOLD = 0.5
THRESHOLD_GRID = np.linspace(0.05, 0.99, 95)

RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Source data  : {SOURCE_DATA}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Device       : {DEVICE}")


Source data  : ..\..\..\Data\CSVs\aeso_merged_2020_2025.csv
Output dir   : random_search_run
Device       : cpu


In [2]:

# 2. Feature definitions
FEATURES = [
    "ACTUAL_POOL_PRICE",
    "ACTUAL_AIL",
    "gas_total", "wind_total", "solar_total", "coal_total", "hydro_total",
    "net_export", "IMPORT_BC", "IMPORT_MT", "IMPORT_SK",
    "net_load", "net_load_3h_change",
    "reserve_margin", "resilience_buffer",
    "renewables_share", "dispatchable_ratio", "gas_ratio",
    "renewable_generation", "dispatchable_generation",
    "sin_hour", "cos_hour",
    "sin_dow", "cos_dow",
    "sin_year_1", "cos_year_1",
    "sin_hour_2", "cos_hour_2",
    "sin_year_2", "cos_year_2",
    "is_weekend", "is_stampede",
]

NO_SCALE = [
    "sin_hour", "cos_hour", "sin_dow", "cos_dow",
    "sin_year_1", "cos_year_1", "sin_hour_2", "cos_hour_2", "sin_year_2", "cos_year_2",
    "is_weekend", "is_stampede",
]
CONTINUOUS = [c for c in FEATURES if c not in NO_SCALE]

BASE_REQUIRED_COLUMNS = [
    "datetime", "ACTUAL_POOL_PRICE", "ACTUAL_AIL",
    "gas_total", "wind_total", "solar_total", "coal_total", "hydro_total",
    "net_export", "IMPORT_BC", "IMPORT_MT", "IMPORT_SK",
    "net_load", "net_load_3h_change", "reserve_margin", "resilience_buffer",
    "renewables_share", "dispatchable_ratio", "gas_ratio",
    "renewable_generation", "dispatchable_generation",
]

print(f"Feature count: {len(FEATURES)}")


Feature count: 32


In [3]:

# 3. Load and clean
df = pd.read_csv(SOURCE_DATA, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

missing_columns = sorted(set(BASE_REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Source data is missing required columns: {missing_columns}")

# Remove Feb 29 so yearly cyclical encoding stays on a 365-day cycle.
df = df[~((df["datetime"].dt.month == 2) & (df["datetime"].dt.day == 29))].reset_index(drop=True)

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")


Rows: 48,839  |  Columns: 108


In [4]:

# 4. Feature engineering
# The target spike_lead_2 already exists in the merged CSV.
# Here we only add calendar features and event flags used by the model.

if "is_weekend" not in df.columns:
    df["is_weekend"] = (df["datetime"].dt.dayofweek >= 5).astype(int)

if "is_stampede" not in df.columns:
    df["is_stampede"] = 0
    for start, end in [
        ("2021-07-09", "2021-07-18"), ("2022-07-08", "2022-07-17"),
        ("2023-07-07", "2023-07-16"), ("2024-07-05", "2024-07-14"),
        ("2025-07-04", "2025-07-13"),
    ]:
        df.loc[df["datetime"].between(start, end), "is_stampede"] = 1

doy = df["datetime"].dt.day_of_year.astype(float)

df["sin_hour"] = np.sin(2 * np.pi * df["datetime"].dt.hour / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["datetime"].dt.hour / 24)
df["sin_dow"] = np.sin(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["cos_dow"] = np.cos(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["sin_year_1"] = np.sin(2 * np.pi * doy / 365)
df["cos_year_1"] = np.cos(2 * np.pi * doy / 365)

df["sin_hour_2"] = np.sin(4 * np.pi * df["datetime"].dt.hour / 24)
df["cos_hour_2"] = np.cos(4 * np.pi * df["datetime"].dt.hour / 24)
df["sin_year_2"] = np.sin(4 * np.pi * doy / 365)
df["cos_year_2"] = np.cos(4 * np.pi * doy / 365)

df = df.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
df[TARGET] = df[TARGET].astype(int)

print(f"Modeling rows : {len(df):,}")
print(f"Spike rate    : {df[TARGET].mean():.4f}")


Modeling rows : 48,839
Spike rate    : 0.0995


In [5]:

# 5. Train / validation / test split
train = df[df["datetime"] < TRAIN_END_EXCLUSIVE].reset_index(drop=True)
validation = df[(df["datetime"] >= TRAIN_END_EXCLUSIVE) & (df["datetime"] < VAL_END_EXCLUSIVE)].reset_index(drop=True)
test = df[df["datetime"] >= TEST_START].reset_index(drop=True)

print(f"Train      : {len(train):,} rows | spike rate = {train[TARGET].mean():.4f}")
print(f"Validation : {len(validation):,} rows | spike rate = {validation[TARGET].mean():.4f}")
print(f"Test       : {len(test):,} rows | spike rate = {test[TARGET].mean():.4f}")


Train      : 33,672 rows | spike rate = 0.1238
Validation : 9,624 rows | spike rate = 0.0614
Test       : 5,543 rows | spike rate = 0.0179


In [6]:

# 6. Sliding-window sequence builder

def make_sequences(X, y, indices, seq_len):
    seqs, labels, used_indices = [], [], []
    for i in indices:
        if i - seq_len + 1 < 0:
            continue
        seqs.append(X[i - seq_len + 1 : i + 1])
        labels.append(y[i])
        used_indices.append(i)
    return (
        np.array(seqs, dtype=np.float32),
        np.array(labels, dtype=np.float32),
        np.array(used_indices, dtype=np.int64),
    )


In [7]:

# 7. Dataset and LSTM model
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :])).squeeze(-1)


## Why F1 score?
With spikes representing roughly 9% of observations, accuracy would be misleading.
F1 is the main selection metric here because it balances precision and recall more evenly.
We still report F2 as a secondary metric because missed spikes still matter economically.


In [8]:
# 8. Small candidate grid
# This pass stays tightly focused on the best region found so far: 48 / 96 / 2.
# The exact Optuna configuration is kept in the list, plus a few nearby variations.
# Each candidate is ranked by its best validation F1 after threshold tuning.

CANDIDATE_GRID = [
    {"seq_len": 48, "hidden_size": 96, "num_layers": 2, "dropout": 0.18, "lr": 0.0016, "weight_decay": 4e-6, "batch_size": 128},
    {"seq_len": 48, "hidden_size": 96, "num_layers": 2, "dropout": 0.20, "lr": 0.0018, "weight_decay": 4e-6, "batch_size": 128},
    {"seq_len": 48, "hidden_size": 96, "num_layers": 2, "dropout": 0.22, "lr": 0.0020, "weight_decay": 4e-6, "batch_size": 128},
    {"seq_len": 48, "hidden_size": 96, "num_layers": 2, "dropout": 0.2368209952651108, "lr": 0.0021576967455896826, "weight_decay": 3.972110727381911e-06, "batch_size": 128},
    {"seq_len": 48, "hidden_size": 96, "num_layers": 2, "dropout": 0.26, "lr": 0.0023, "weight_decay": 5e-6, "batch_size": 128},
]

pretest = pd.concat([train, validation], ignore_index=True)
y_tv = pretest[TARGET].values.astype(np.float32)
tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
folds = list(tscv.split(pretest))
tr_tune_idx, va_tune_idx = folds[-1]

best_f1 = -1.0
BEST = None
search_results = []

for trial, params in enumerate(CANDIDATE_GRID, start=1):
    scaler = StandardScaler()
    scaler.fit(pretest.iloc[tr_tune_idx][CONTINUOUS])
    X_tv_sc = pretest[FEATURES].copy().astype(float)
    X_tv_sc[CONTINUOUS] = scaler.transform(X_tv_sc[CONTINUOUS])
    X_tv_sc = X_tv_sc.values.astype(np.float32)

    X_tr, y_tr, _ = make_sequences(X_tv_sc, y_tv, tr_tune_idx, params["seq_len"])
    X_va, y_va, _ = make_sequences(X_tv_sc, y_tv, va_tune_idx, params["seq_len"])

    tr_loader = DataLoader(
        SeqDataset(X_tr, y_tr),
        batch_size=params["batch_size"],
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    va_loader = DataLoader(
        SeqDataset(X_va, y_va),
        batch_size=params["batch_size"] * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    n_pos = float(y_tr.sum())
    n_neg = float(len(y_tr) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    model = LSTMClassifier(len(FEATURES), params["hidden_size"], params["num_layers"], params["dropout"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    trial_best_f1 = -1.0
    trial_best_f2 = -1.0
    trial_best_threshold = THRESHOLD
    patience_ctr = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for X_batch, y_batch in tr_loader:
            optimizer.zero_grad()
            criterion(model(X_batch.to(DEVICE)), y_batch.to(DEVICE)).backward()
            optimizer.step()

        model.eval()
        probs = []
        with torch.no_grad():
            for X_batch, _ in va_loader:
                probs.append(torch.sigmoid(model(X_batch.to(DEVICE))).cpu().numpy())
        probs = np.concatenate(probs)

        epoch_rows = []
        for threshold in THRESHOLD_GRID:
            preds = (probs >= threshold).astype(int)
            epoch_rows.append(
                {
                    "threshold": float(threshold),
                    "f1": float(f1_score(y_va, preds, zero_division=0)),
                    "f2": float(fbeta_score(y_va, preds, beta=2, zero_division=0)),
                    "precision": float(precision_score(y_va, preds, zero_division=0)),
                }
            )

        epoch_best = pd.DataFrame(epoch_rows).sort_values(
            ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
        ).reset_index(drop=True).iloc[0]

        if float(epoch_best["f1"]) > trial_best_f1:
            trial_best_f1 = float(epoch_best["f1"])
            trial_best_f2 = float(epoch_best["f2"])
            trial_best_threshold = float(epoch_best["threshold"])
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    search_results.append(
        {
            "trial": trial,
            **params,
            "best_threshold": round(float(trial_best_threshold), 2),
            "best_f1": round(float(trial_best_f1), 4),
            "best_f2_at_best_f1_threshold": round(float(trial_best_f2), 4),
        }
    )
    print(
        f"Trial {trial:2d}/{len(CANDIDATE_GRID)}  "
        f"F1={trial_best_f1:.4f}  F2={trial_best_f2:.4f}  threshold={trial_best_threshold:.2f}  {params}"
    )

    if trial_best_f1 > best_f1:
        best_f1 = trial_best_f1
        BEST = params

search_results_df = pd.DataFrame(search_results).sort_values("best_f1", ascending=False).reset_index(drop=True)
print()
print(f"Best config  : {BEST}")
print(f"Best tune F1 : {best_f1:.4f}")
search_results_df


Trial  1/5  F1=0.5910  F2=0.5633  threshold=0.92  {'seq_len': 48, 'hidden_size': 96, 'num_layers': 2, 'dropout': 0.18, 'lr': 0.0016, 'weight_decay': 4e-06, 'batch_size': 128}


Trial  2/5  F1=0.5823  F2=0.6150  threshold=0.87  {'seq_len': 48, 'hidden_size': 96, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0018, 'weight_decay': 4e-06, 'batch_size': 128}


Trial  3/5  F1=0.5734  F2=0.5392  threshold=0.91  {'seq_len': 48, 'hidden_size': 96, 'num_layers': 2, 'dropout': 0.22, 'lr': 0.002, 'weight_decay': 4e-06, 'batch_size': 128}


Trial  4/5  F1=0.5827  F2=0.5886  threshold=0.91  {'seq_len': 48, 'hidden_size': 96, 'num_layers': 2, 'dropout': 0.2368209952651108, 'lr': 0.0021576967455896826, 'weight_decay': 3.972110727381911e-06, 'batch_size': 128}


Trial  5/5  F1=0.5978  F2=0.5871  threshold=0.93  {'seq_len': 48, 'hidden_size': 96, 'num_layers': 2, 'dropout': 0.26, 'lr': 0.0023, 'weight_decay': 5e-06, 'batch_size': 128}

Best config  : {'seq_len': 48, 'hidden_size': 96, 'num_layers': 2, 'dropout': 0.26, 'lr': 0.0023, 'weight_decay': 5e-06, 'batch_size': 128}
Best tune F1 : 0.5978


,trial,seq_len,hidden_size,num_layers,dropout,lr,weight_decay,batch_size,best_threshold,best_f1,best_f2_at_best_f1_threshold
0,5,48,96,2,0.260000,0.002300,0.000005,128,0.93,0.5978,0.5871
1,1,48,96,2,0.180000,0.001600,0.000004,128,0.92,0.5910,0.5633
2,4,48,96,2,0.236821,0.002158,0.000004,128,0.91,0.5827,0.5886
3,2,48,96,2,0.200000,0.001800,0.000004,128,0.87,0.5823,0.6150
4,3,48,96,2,0.220000,0.002000,0.000004,128,0.91,0.5734,0.5392


In [9]:
# 9. Full TimeSeriesSplit cross-validation with the best configuration
# We keep the same idea here: each fold is judged by the best validation F1 after threshold tuning.
pretest = pd.concat([train, validation], ignore_index=True)
y_tv = pretest[TARGET].values.astype(np.float32)
fold_f1s = []
fold_f2s_at_best_f1_threshold = []
fold_thresholds = []

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    scaler = StandardScaler()
    scaler.fit(pretest.iloc[tr_idx][CONTINUOUS])
    X_tv_sc = pretest[FEATURES].copy().astype(float)
    X_tv_sc[CONTINUOUS] = scaler.transform(X_tv_sc[CONTINUOUS])
    X_tv_sc = X_tv_sc.values.astype(np.float32)

    X_tr, y_tr, _ = make_sequences(X_tv_sc, y_tv, tr_idx, BEST["seq_len"])
    X_va, y_va, _ = make_sequences(X_tv_sc, y_tv, va_idx, BEST["seq_len"])

    tr_loader = DataLoader(
        SeqDataset(X_tr, y_tr),
        batch_size=BEST["batch_size"],
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    va_loader = DataLoader(
        SeqDataset(X_va, y_va),
        batch_size=BEST["batch_size"] * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    n_pos = float(y_tr.sum())
    n_neg = float(len(y_tr) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    model = LSTMClassifier(len(FEATURES), BEST["hidden_size"], BEST["num_layers"], BEST["dropout"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=BEST["lr"], weight_decay=BEST["weight_decay"])

    best_fold_f1 = -1.0
    best_fold_f2 = -1.0
    best_fold_threshold = THRESHOLD
    patience_ctr = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for X_batch, y_batch in tr_loader:
            optimizer.zero_grad()
            criterion(model(X_batch.to(DEVICE)), y_batch.to(DEVICE)).backward()
            optimizer.step()

        model.eval()
        probs = []
        with torch.no_grad():
            for X_batch, _ in va_loader:
                probs.append(torch.sigmoid(model(X_batch.to(DEVICE))).cpu().numpy())
        probs = np.concatenate(probs)

        epoch_rows = []
        for threshold in THRESHOLD_GRID:
            preds = (probs >= threshold).astype(int)
            epoch_rows.append(
                {
                    "threshold": float(threshold),
                    "f1": float(f1_score(y_va, preds, zero_division=0)),
                    "f2": float(fbeta_score(y_va, preds, beta=2, zero_division=0)),
                    "precision": float(precision_score(y_va, preds, zero_division=0)),
                }
            )

        epoch_best = pd.DataFrame(epoch_rows).sort_values(
            ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
        ).reset_index(drop=True).iloc[0]

        if float(epoch_best["f1"]) > best_fold_f1:
            best_fold_f1 = float(epoch_best["f1"])
            best_fold_f2 = float(epoch_best["f2"])
            best_fold_threshold = float(epoch_best["threshold"])
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    fold_f1s.append(best_fold_f1)
    fold_f2s_at_best_f1_threshold.append(best_fold_f2)
    fold_thresholds.append(best_fold_threshold)
    print(
        f"Fold {fold}/{N_CV_SPLITS}  best F1 = {best_fold_f1:.4f}  "
        f"F2 = {best_fold_f2:.4f}  threshold = {best_fold_threshold:.2f}  (stopped epoch {epoch})"
    )

print()
print(f"Mean CV F1 : {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")
print(
    f"Mean CV F2 : {np.mean(fold_f2s_at_best_f1_threshold):.4f} +/- {np.std(fold_f2s_at_best_f1_threshold):.4f}  "
    f"(at the best-F1 thresholds)"
)


Fold 1/5  best F1 = 0.5911  F2 = 0.5548  threshold = 0.93  (stopped epoch 7)


Fold 2/5  best F1 = 0.5450  F2 = 0.5350  threshold = 0.84  (stopped epoch 7)


Fold 3/5  best F1 = 0.6529  F2 = 0.7023  threshold = 0.58  (stopped epoch 10)


Fold 4/5  best F1 = 0.6684  F2 = 0.6466  threshold = 0.93  (stopped epoch 8)


Fold 5/5  best F1 = 0.5727  F2 = 0.5754  threshold = 0.89  (stopped epoch 7)

Mean CV F1 : 0.6060 +/- 0.0472
Mean CV F2 : 0.6028 +/- 0.0624  (at the best-F1 thresholds)


In [10]:
# 10. Final model: train on train, early stop on validation using tuned F1,
# then apply the best validation threshold once to the untouched test set.
full = pd.concat([train, validation, test], ignore_index=True)
y_full = full[TARGET].values.astype(np.float32)

train_idx = np.flatnonzero(full["datetime"] < TRAIN_END_EXCLUSIVE)
val_idx = np.flatnonzero((full["datetime"] >= TRAIN_END_EXCLUSIVE) & (full["datetime"] < VAL_END_EXCLUSIVE))
test_idx = np.flatnonzero(full["datetime"] >= TEST_START)

scaler_final = StandardScaler()
scaler_final.fit(train[CONTINUOUS])
X_full = full[FEATURES].copy().astype(float)
X_full[CONTINUOUS] = scaler_final.transform(X_full[CONTINUOUS])
X_full = X_full.values.astype(np.float32)

X_tr, y_tr, tr_used_idx = make_sequences(X_full, y_full, train_idx, BEST["seq_len"])
X_va, y_va, va_used_idx = make_sequences(X_full, y_full, val_idx, BEST["seq_len"])
X_te, y_te, te_used_idx = make_sequences(X_full, y_full, test_idx, BEST["seq_len"])

tr_loader = DataLoader(
    SeqDataset(X_tr, y_tr),
    batch_size=BEST["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
va_loader = DataLoader(
    SeqDataset(X_va, y_va),
    batch_size=BEST["batch_size"] * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
te_loader = DataLoader(
    SeqDataset(X_te, y_te),
    batch_size=BEST["batch_size"] * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

n_pos = float(y_tr.sum())
n_neg = float(len(y_tr) - n_pos)
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

final_model = LSTMClassifier(len(FEATURES), BEST["hidden_size"], BEST["num_layers"], BEST["dropout"]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(final_model.parameters(), lr=BEST["lr"], weight_decay=BEST["weight_decay"])

best_val_f1_tuned = -1.0
best_val_f2_tuned = -1.0
best_val_threshold_tuned = THRESHOLD
best_state = None
patience_ctr = 0
training_history = []

for epoch in range(1, MAX_EPOCHS + 1):
    final_model.train()
    batch_losses = []
    for X_batch, y_batch in tr_loader:
        optimizer.zero_grad()
        loss = criterion(final_model(X_batch.to(DEVICE)), y_batch.to(DEVICE))
        loss.backward()
        optimizer.step()
        batch_losses.append(float(loss.item()))

    torch.save(final_model.state_dict(), CHECKPOINT_DIR / "latest_model.pt")

    final_model.eval()
    val_probs = []
    with torch.no_grad():
        for X_batch, _ in va_loader:
            val_probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
    val_probs = np.concatenate(val_probs)

    epoch_rows = []
    for threshold in THRESHOLD_GRID:
        preds = (val_probs >= threshold).astype(int)
        epoch_rows.append(
            {
                "threshold": float(threshold),
                "f1": float(f1_score(y_va, preds, zero_division=0)),
                "f2": float(fbeta_score(y_va, preds, beta=2, zero_division=0)),
                "precision": float(precision_score(y_va, preds, zero_division=0)),
                "recall": float(recall_score(y_va, preds, zero_division=0)),
                "predicted_positive_rate": float(preds.mean()),
            }
        )

    epoch_best = pd.DataFrame(epoch_rows).sort_values(
        ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
    ).reset_index(drop=True).iloc[0]

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(batch_losses)) if batch_losses else np.nan,
            "validation_best_f1": float(epoch_best["f1"]),
            "validation_best_f2": float(epoch_best["f2"]),
            "validation_best_threshold": float(epoch_best["threshold"]),
        }
    )

    if float(epoch_best["f1"]) > best_val_f1_tuned:
        best_val_f1_tuned = float(epoch_best["f1"])
        best_val_f2_tuned = float(epoch_best["f2"])
        best_val_threshold_tuned = float(epoch_best["threshold"])
        best_state = {k: v.cpu().clone() for k, v in final_model.state_dict().items()}
        torch.save(best_state, CHECKPOINT_DIR / "best_model.pt")
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            break

final_model.load_state_dict(best_state)

final_model.eval()
val_probs = []
with torch.no_grad():
    for X_batch, _ in va_loader:
        val_probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
val_probs = np.concatenate(val_probs)

# This is the one threshold table we keep and save for later inspection.
threshold_rows = []
for threshold in THRESHOLD_GRID:
    val_preds = (val_probs >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": round(float(threshold), 2),
            "f1": float(f1_score(y_va, val_preds, zero_division=0)),
            "f2": float(fbeta_score(y_va, val_preds, beta=2, zero_division=0)),
            "precision": float(precision_score(y_va, val_preds, zero_division=0)),
            "recall": float(recall_score(y_va, val_preds, zero_division=0)),
            "predicted_positive_rate": float(val_preds.mean()),
        }
    )

threshold_search_df = pd.DataFrame(threshold_rows).sort_values(
    ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
).reset_index(drop=True)

best_threshold_row = threshold_search_df.iloc[0]
SELECTED_THRESHOLD = float(best_threshold_row["threshold"])
validation_threshold_metrics = {
    "threshold": float(best_threshold_row["threshold"]),
    "f1": float(best_threshold_row["f1"]),
    "f2": float(best_threshold_row["f2"]),
    "precision": float(best_threshold_row["precision"]),
    "recall": float(best_threshold_row["recall"]),
    "predicted_positive_rate": float(best_threshold_row["predicted_positive_rate"]),
}

probs = []
with torch.no_grad():
    for X_batch, _ in te_loader:
        probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
probs = np.concatenate(probs)
preds = (probs >= SELECTED_THRESHOLD).astype(int)

test_metrics = {
    "f1": float(f1_score(y_te, preds, zero_division=0)),
    "f2": float(fbeta_score(y_te, preds, beta=2, zero_division=0)),
    "precision": float(precision_score(y_te, preds, zero_division=0)),
    "recall": float(recall_score(y_te, preds, zero_division=0)),
    "pr_auc": float(average_precision_score(y_te, probs)),
    "roc_auc": float(roc_auc_score(y_te, probs)),
}

print()
print(f"Best validation F1 during training : {best_val_f1_tuned:.4f}")
print(f"Best validation threshold seen     : {best_val_threshold_tuned:.2f}")
print()
print(f"Selected validation threshold (max F1): {SELECTED_THRESHOLD:.2f}")
print(f"Validation F1 at selected threshold   : {validation_threshold_metrics['f1']:.4f}")
print(f"Validation F2 at selected threshold   : {validation_threshold_metrics['f2']:.4f}")
print()
print(f"Test metrics (threshold = {SELECTED_THRESHOLD:.2f})")
for metric_name, metric_value in test_metrics.items():
    print(f"  {metric_name:<9}: {metric_value:.4f}")

test_predictions = full.iloc[te_used_idx][["datetime", "ACTUAL_POOL_PRICE"]].copy()
test_predictions["y_true"] = y_te.astype(int)
test_predictions["y_prob"] = probs
test_predictions["y_pred"] = preds



Best validation F1 during training : 0.5993
Best validation threshold seen     : 0.89

Selected validation threshold (max F1): 0.89
Validation F1 at selected threshold   : 0.5993
Validation F2 at selected threshold   : 0.5857

Test metrics (threshold = 0.89)
  f1       : 0.3772
  f2       : 0.4095
  precision: 0.3333
  recall   : 0.4343
  pr_auc   : 0.3260
  roc_auc  : 0.9020


In [11]:
# 11. Approximate top-20 feature importance using validation permutation importance
# If shuffling a feature hurts validation F1, that feature is helping the model.
# If shuffling a feature improves validation F1, that feature is probably noisy or harmful.

baseline_val_preds = (val_probs >= SELECTED_THRESHOLD).astype(int)
baseline_val_f1 = f1_score(y_va, baseline_val_preds, zero_division=0)
baseline_val_f2 = fbeta_score(y_va, baseline_val_preds, beta=2, zero_division=0)

rng = np.random.default_rng(RANDOM_SEED)
importance_rows = []
n_repeats = 3

for feature_index, feature_name in enumerate(FEATURES):
    repeat_f1_scores = []
    repeat_f2_scores = []
    for _ in range(n_repeats):
        X_perm = X_va.copy()
        shuffled_order = rng.permutation(len(X_perm))
        X_perm[:, :, feature_index] = X_perm[shuffled_order, :, feature_index]

        perm_loader = DataLoader(
            SeqDataset(X_perm, y_va),
            batch_size=BEST["batch_size"] * 2,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

        perm_probs = []
        with torch.no_grad():
            for X_batch, _ in perm_loader:
                perm_probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
        perm_probs = np.concatenate(perm_probs)
        perm_preds = (perm_probs >= SELECTED_THRESHOLD).astype(int)
        repeat_f1_scores.append(f1_score(y_va, perm_preds, zero_division=0))
        repeat_f2_scores.append(fbeta_score(y_va, perm_preds, beta=2, zero_division=0))

    mean_perm_f1 = float(np.mean(repeat_f1_scores))
    mean_perm_f2 = float(np.mean(repeat_f2_scores))
    signed_change = float(baseline_val_f1 - mean_perm_f1)

    importance_rows.append(
        {
            "feature": feature_name,
            "baseline_f1": float(baseline_val_f1),
            "permuted_f1": mean_perm_f1,
            "baseline_f2": float(baseline_val_f2),
            "permuted_f2": mean_perm_f2,
            "signed_importance_f1_change": signed_change,
            "absolute_importance_f1_change": float(abs(signed_change)),
            "impact_sign": "+" if signed_change > 0 else "-" if signed_change < 0 else "0",
            "impact_direction": "helps" if signed_change > 0 else "hurts" if signed_change < 0 else "neutral",
        }
    )

feature_importance_df = pd.DataFrame(importance_rows).sort_values(
    "absolute_importance_f1_change", ascending=False
).reset_index(drop=True)

top_20_feature_importance = feature_importance_df.head(20)
print("Top 20 approximate feature importances (ordered by absolute F1 impact):")
top_20_feature_importance


Top 20 approximate feature importances (ordered by absolute F1 impact):


,feature,baseline_f1,permuted_f1,baseline_f2,permuted_f2,signed_importance_f1_change,absolute_importance_f1_change,impact_sign,impact_direction
0,ACTUAL_POOL_PRICE,0.599297,0.287088,0.585709,0.235126,0.312209,0.312209,+,helps
1,solar_total,0.599297,0.552269,0.585709,0.550853,0.047028,0.047028,+,helps
2,wind_total,0.599297,0.554252,0.585709,0.502219,0.045045,0.045045,+,helps
3,sin_hour_2,0.599297,0.561160,0.585709,0.546372,0.038137,0.038137,+,helps
4,net_load_3h_change,0.599297,0.562427,0.585709,0.532577,0.036870,0.036870,+,helps
5,net_load,0.599297,0.563875,0.585709,0.507275,0.035422,0.035422,+,helps
6,renewables_share,0.599297,0.572668,0.585709,0.528793,0.026629,0.026629,+,helps
7,sin_hour,0.599297,0.573410,0.585709,0.563599,0.025887,0.025887,+,helps
8,dispatchable_generation,0.599297,0.579294,0.585709,0.536284,0.020003,0.020003,+,helps
9,renewable_generation,0.599297,0.581848,0.585709,0.541995,0.017449,0.017449,+,helps


In [12]:
# 12. Save outputs
training_history_df = pd.DataFrame(training_history)

joblib.dump(scaler_final, OUTPUT_DIR / "scaler.joblib")
torch.save(final_model.state_dict(), OUTPUT_DIR / "best_model.pt")

latest_checkpoint = CHECKPOINT_DIR / "latest_model.pt"
if latest_checkpoint.exists():
    (OUTPUT_DIR / "latest_model.pt").write_bytes(latest_checkpoint.read_bytes())

pd.DataFrame(
    {
        "cv_fold_f1": fold_f1s,
        "cv_fold_f2_at_best_f1_threshold": fold_f2s_at_best_f1_threshold,
        "cv_best_threshold": fold_thresholds,
    }
).to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
search_results_df.to_csv(OUTPUT_DIR / "search_results.csv", index=False)
threshold_search_df.to_csv(OUTPUT_DIR / "validation_threshold_search.csv", index=False)
feature_importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
top_20_feature_importance.to_csv(OUTPUT_DIR / "top_20_feature_importance.csv", index=False)
training_history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_params": BEST,
            "candidate_grid": CANDIDATE_GRID,
            "cv_fold_f1s": [round(float(x), 4) for x in fold_f1s],
            "cv_mean_f1": round(float(np.mean(fold_f1s)), 4),
            "cv_std_f1": round(float(np.std(fold_f1s)), 4),
            "cv_fold_f2s_at_best_f1_threshold": [round(float(x), 4) for x in fold_f2s_at_best_f1_threshold],
            "cv_mean_f2_at_best_f1_threshold": round(float(np.mean(fold_f2s_at_best_f1_threshold)), 4),
            "cv_std_f2_at_best_f1_threshold": round(float(np.std(fold_f2s_at_best_f1_threshold)), 4),
            "cv_best_thresholds": [round(float(x), 2) for x in fold_thresholds],
            "best_validation_f1_with_tuned_threshold": round(float(best_val_f1_tuned), 4),
            "best_validation_f2_with_tuned_threshold": round(float(best_val_f2_tuned), 4),
            "best_validation_threshold_during_training": round(float(best_val_threshold_tuned), 4),
            "threshold_selection_metric": "f1",
            "selected_threshold": round(float(SELECTED_THRESHOLD), 4),
            "validation_threshold_metrics": {
                key: round(float(value), 4) for key, value in validation_threshold_metrics.items()
            },
            "test": {
                key: round(float(value), 4) for key, value in test_metrics.items()
            },
        },
        f,
        indent=2,
    )

print()
print(f"Outputs saved to: {OUTPUT_DIR}")



Outputs saved to: random_search_run
